In [7]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

from langgraph.graph.state import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
import sqlite3

from typing import Annotated, TypedDict, List, Dict, Literal
from dotenv import load_dotenv
import os
load_dotenv()

llm_endpoint = HuggingFaceEndpoint(
    repo_id=os.getenv('hf_model'),
    provider='together'
    )
model = ChatHuggingFace(llm=llm_endpoint)

google_model = ChatGoogleGenerativeAI(model='gemini-3.1-flash-lite')


In [5]:
search_tool = DuckDuckGoSearchRun()

In [8]:
@tool
def calculator(first_num:float, second_num:float,operator:Literal['add','sub','mul','div'])->dict:
    """
    A function to perform math operations - add, subract, multiply, divide
    """
    result = 0.0
    if operator == 'add':
        result = first_num + second_num
    elif operator == 'sub':
        result = first_num - second_num
    elif operator == 'mul':
        result = first_num * second_num
    elif operator == 'div':
        if second_num == 0:
            return {'error':'cannot divide with zero'}
        else:
            result = first_num / second_num
    else:
        return {'error':f'unsupported operation{operator}'}

    return {'first_num':first_num,'second_num':second_num,'operator':operator,'result':result}



In [9]:
calculator.invoke({'first_num':5,'second_num':6,'operator':'add'})

{'first_num': 5.0, 'second_num': 6.0, 'operator': 'add', 'result': 11.0}

In [ ]:
class ChatSchema(TypedDict):
    topic:str
    messages:Annotated[List[BaseMessage], add_messages]


def chat_node(state:ChatSchema):
    messages = state['messages']
    response = google_model.invoke(messages)
    return {'messages':[AIMessage(content=response.content)]}
